In [1]:
# Use Pyspark2 kernal because there are some bug with cache in Pyspark3

from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession.builder\
.appName("Jewel:Caching spark table")\
.config('spark.ui.port', '0')\
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse")\
.enableHiveSupport().master('yarn').getOrCreate()

In [2]:
orders_schema = "orders_id long, orders_date date, customer_id long, orders_status string"
# This is for pyspark3
orders_df = spark.read.csv("/public/trendytech/orders/orders_1gb.csv", header=True, schema=orders_schema)
# This is for pyspark2
# orders_df = spark.read.format("csv").schema(orders_schema).load("/public/trendytech/orders/orders_1gb.csv")
# orders_df.show(5)

In [3]:
orders_df.show(5)

+---------+-----------+-----------+---------------+
|orders_id|orders_date|customer_id|  orders_status|
+---------+-----------+-----------+---------------+
|        2| 2013-07-25|        256|PENDING_PAYMENT|
|        3| 2013-07-25|      12111|       COMPLETE|
|        4| 2013-07-25|       8827|         CLOSED|
|        5| 2013-07-25|      11318|       COMPLETE|
|        6| 2013-07-25|       7130|       COMPLETE|
+---------+-----------+-----------+---------------+
only showing top 5 rows



In [4]:
spark.sql(f"create database if not exists {username}_caching_spark")

""


In [5]:
orders_df.write.mode("ignore").format("csv").saveAsTable(f"{username}_caching_spark.tbl_orders")

In [6]:
spark.sql(f"describe extended {username}_caching_spark.tbl_orders").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           orders_id|              bigint|   null|
|         orders_date|                date|   null|
|         customer_id|              bigint|   null|
|       orders_status|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|            Database|itv027775_caching...|       |
|               Table|          tbl_orders|       |
|               Owner|           itv027775|       |
|        Created Time|Thu Aug 27 13:59:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.1.2|       |
|                Type|             MANAGED|       |
|            Provider|                 csv|       |
|          Statistics|     840836631 bytes|       |
|            Location|hdfs://m01.itvers...|       |
|       Serd

In [7]:
spark.sql(f"select count(*) number_of_records from {username}_caching_spark.tbl_orders").show()

+-----------------+
|number_of_records|
+-----------------+
|         25831125|
+-----------------+



In [8]:
spark.sql(f"cache table {username}_caching_spark.tbl_orders")

""


In [16]:
spark.sql(f"select count(*) number_of_records from {username}_caching_spark.tbl_orders").show()

+-----------------+
|number_of_records|
+-----------------+
|         25831124|
+-----------------+



In [10]:
spark.sql(f"select distinct orders_status from {username}_caching_spark.tbl_orders").show()

+---------------+
|  orders_status|
+---------------+
|PENDING_PAYMENT|
|       COMPLETE|
|        ON_HOLD|
| PAYMENT_REVIEW|
|     PROCESSING|
|         CLOSED|
|SUSPECTED_FRAUD|
|        PENDING|
|       CANCELED|
+---------------+



In [18]:
spark.sql(f"select count(distinct orders_status) distinct_count from {username}_caching_spark.tbl_orders").show()

+--------------+
|distinct_count|
+--------------+
|            10|
+--------------+



In [14]:
spark.sql(f"uncache table {username}_caching_spark.tbl_orders")

""


In [7]:
spark.sql(f"cache lazy table {username}_caching_spark.tbl_orders")

""


In [19]:
spark.sql(f"select orders_status, count(*) from {username}_caching_spark.tbl_orders group by orders_status").show()

+---------------+--------+
|  orders_status|count(1)|
+---------------+--------+
|PENDING_PAYMENT| 5636250|
|       COMPLETE| 8587125|
|        ON_HOLD| 1424250|
| PAYMENT_REVIEW|  273375|
|         BOOKED|       1|
|     PROCESSING| 3103125|
|         CLOSED| 2833499|
|SUSPECTED_FRAUD|  584250|
|        PENDING| 2853750|
|       CANCELED|  535500|
+---------------+--------+



In [16]:
spark.sql(f"insert into {username}_caching_spark.tbl_orders values(111111, CAST('2023-05-29' as DATE), 222222, 'BOOKED')")

""


In [9]:
spark.sql("clear cache") #it clears all cached table(s)

""


In [10]:
spark.catalog.currentDatabase()

'default'

In [26]:
spark.sql(f"use {username}_caching_spark")

""


In [27]:
spark.catalog.isCached(f"{username}_caching_spark.tbl_orders")

False

In [29]:
spark.catalog.clearCache()

In [4]:
spark.sql(f"create database if not exists {username}_caching_demo")

""


In [5]:
!ls /user/itv027775/orders

ls: cannot access /user/itv027775/orders: No such file or directory


In [ ]:
spark.sql(f"create table {username}_caching_demo.tbl_order_ext(order_id long, order_date string, "\
          +"custorme_id long, order_status string) using csv location '/user/itv'")